In [ ]:
import pandas as pd
import joblib  # For loading .pkl files
import requests  # For making API calls
import time  # For retrying API calls

# Load the trained Random Forest model, scaler, and label encoder
rf_model = joblib.load('rf_model.pkl')  # Replace with the correct path if needed
scaler = joblib.load('scaler.pkl')     # Replace with the correct path if needed
label_encoder = joblib.load('label_encoder.pkl')  # Replace with the correct path if needed

# Blynk API URLs for real-time sensor values
BLYNK_TOKEN = os.getenv("BLYNK_TOKEN", "")  # Replace with your actual token
pH_URL = f"https://blynk.cloud/external/api/get?token={BLYNK_TOKEN}&V0"
TDS_URL = f"https://blynk.cloud/external/api/get?token={BLYNK_TOKEN}&V1"
Turbidity_URL = f"https://blynk.cloud/external/api/get?token={BLYNK_TOKEN}&V2"
Temperature_URL = f"https://blynk.cloud/external/api/get?token={BLYNK_TOKEN}&V3"

# Blynk API URL for result upload
RESULT_UPLOAD_URL = f"https://blynk.cloud/external/api/update?token={BLYNK_TOKEN}&V6={{}}"

def get_sensor_value(url, retries=3):
    """Fetches sensor data from the API, with retries if the response is invalid."""
    for _ in range(retries):
        response = requests.get(url)
        if response.status_code == 200:
            data = response.text.strip()
            if data.replace('.', '', 1).isdigit():  # Check if it's a valid number
                return float(data)
        print(f"Retrying fetching data from {url}...")
        time.sleep(2)  # Wait before retrying
    raise Exception(f"Failed to fetch valid data from {url}")

def upload_result_to_blynk(result):
    """Uploads prediction results to Blynk API."""
    response = requests.get(RESULT_UPLOAD_URL.format(result))
    if response.status_code == 200:
        print(f"Result '{result}' successfully uploaded to Blynk.")
    else:
        raise Exception(f"Failed to upload result to Blynk. HTTP Status Code: {response.status_code}")

def disaster(pH, TDS, Turbidity, Temperature):
    """Predicts disaster based on sensor values using the trained model."""
    new_data = pd.DataFrame({
        'pH': [pH],
        'TDS': [TDS],
        'Turbidity': [Turbidity],
        'Temperature': [Temperature]
    })

    # Scale the new data using the loaded scaler
    new_data_scaled = scaler.transform(new_data)

    # Make predictions using the loaded Random Forest model
    dt_prediction = rf_model.predict(new_data_scaled)

    # Convert numerical prediction back to the original label
    result = label_encoder.inverse_transform(dt_prediction)

    return result[0]

# Main function to fetch real-time sensor values, make a prediction, and upload the result
if __name__ == "__main__":
    try:
        print("Fetching real-time sensor values...")

        # Fetch real-time sensor values
        pH = get_sensor_value(pH_URL)
        TDS = get_sensor_value(TDS_URL)
        Turbidity = get_sensor_value(Turbidity_URL)
        Temperature = get_sensor_value(Temperature_URL)

        print(f"pH: {pH}")
        print(f"TDS: {TDS}")
        print(f"Turbidity: {Turbidity}")
        print(f"Temperature: {Temperature}")

        # Predict the disaster type
        predicted_result = disaster(pH, TDS, Turbidity, Temperature)

        # Display the prediction
        print("\nPredicted Result:", predicted_result)

        # Upload the prediction result to Blynk
        upload_result_to_blynk(predicted_result)

    except Exception as e:
        print("\nError:", str(e))
